## Project 

*Members: Melissa Cai Shi (BOA), Cassandre Korvink (GATOR), Mikyung Oh (JAGUAR)*

In [ ]:
-- Set Context
USE ROLE ROLE_TEAM_CLAWJAWCOIL;
USE WAREHOUSE ANIMAL_TASK_WH;
USE DATABASE DB_TEAM_CLAWJAWCOIL;

### Task 1: Create 3 schemas in your project database: Bronze, Silver, Gold

In [ ]:
-- Create the 3 medallion layer schemas
CREATE SCHEMA IF NOT EXISTS BRONZE;
CREATE SCHEMA IF NOT EXISTS SILVER;
CREATE SCHEMA IF NOT EXISTS GOLD;

### Task 2: Import the selected dataset into a bronze layer in one or more tables. (This will be your raw data without any modifications.)

In [ ]:
-- Create Bronze layer table design for raw landslide data
CREATE OR REPLACE TABLE BRONZE.LANDSLIDE_RAW (
    source_name                 STRING,
    source_link                 STRING,
    event_id                    STRING,
    event_date                  STRING,
    event_title                 STRING,
    event_description           STRING,
    location_description        STRING,
    location_accuracy           STRING,
    landslide_category          STRING,
    landslide_trigger           STRING,
    landslide_size              STRING,
    landslide_setting           STRING,
    fatality_count              STRING,
    injury_count                STRING,
    storm_name                  STRING,
    photo_link                  STRING,
    notes                       STRING,
    country_name                STRING,
    country_code                STRING,
    admin_division_name         STRING,
    admin_division_population   STRING,
    gazeteer_closest_point      STRING,
    gazeteer_distance           STRING,
    submitted_date              STRING,
    created_date                STRING,
    last_edited_date            STRING,
    longitude                   STRING,
    latitude                    STRING
);


-- Create a stage to load the data in the Bronze layer
USE SCHEMA BRONZE;
CREATE STAGE IF NOT EXISTS OUR_LOAD_STAGE;
--DROP STAGE IF EXISTS OUR_LOAD_STAGE;

In [ ]:
-- Before running this code load the raw data file to the stage
-- Home -> Ingestion -> Add data -> Load files into a Stage

-- Load data into Bronze level
COPY INTO LANDSLIDE_RAW
FROM @OUR_LOAD_STAGE/Global_Landslide_Data_Raw.csv
FILE_FORMAT = (TYPE = 'CSV' FIELD_OPTIONALLY_ENCLOSED_BY = '"' SKIP_HEADER = 1)
ON_ERROR = 'CONTINUE';

In [ ]:
USE DATABASE DB_TEAM_CLAWJAWCOIL;

-- Verify the load
SELECT * 
FROM BRONZE.LANDSLIDE_RAW 
LIMIT 10;

### Task 3-4: Identify and design the appropriate Dimension, Facts and Composite tables needed in your silver layer of your data warehouse.
### Write appropriate commands in an SQL sheet to create Silver layer schema objects.

(See excel sheet for silver layer star design)

In [ ]:
-- Date dimension (supports event_date and other dates)
CREATE OR REPLACE TABLE SILVER.DIM_DATE (
  DATE_ID       INTEGER AUTOINCREMENT PRIMARY KEY,
  DATE          DATE,
  YEAR          INTEGER,
  MONTH         INTEGER,
  DAY           INTEGER,
  DAY_OF_WEEK   VARCHAR,
  QUARTER       VARCHAR
);

-- Source dimension
CREATE OR REPLACE TABLE SILVER.DIM_SOURCE (
  SOURCE_ID     INTEGER AUTOINCREMENT PRIMARY KEY,
  SOURCE_NAME   VARCHAR
);

-- Country dimension
CREATE OR REPLACE TABLE SILVER.DIM_COUNTRY (
  COUNTRY_ID    INTEGER AUTOINCREMENT PRIMARY KEY,
  COUNTRY_NAME  VARCHAR,
  COUNTRY_CODE  VARCHAR
);

-- Admin division (province/state) dimension
CREATE OR REPLACE TABLE SILVER.DIM_ADMIN_DIVISION (
  ADMIN_DIVISION_ID INTEGER AUTOINCREMENT PRIMARY KEY,
  ADMIN_DIVISION_NAME VARCHAR
);

-- Distance / accuracy dimension 
CREATE OR REPLACE TABLE SILVER.DIM_DISTANCE (
  DISTANCE_ID   INTEGER AUTOINCREMENT PRIMARY KEY,
  LOCATION_ACCURACY VARCHAR
);

-- Landslide category dimension
CREATE OR REPLACE TABLE SILVER.DIM_LANDSLIDE_CATEGORY (
  CATEGORY_ID  INTEGER AUTOINCREMENT PRIMARY KEY,
  CATEGORY     VARCHAR
);

-- Landslide trigger dimension
CREATE OR REPLACE TABLE SILVER.DIM_LANDSLIDE_TRIGGER (
  TRIGGER_ID  INTEGER AUTOINCREMENT PRIMARY KEY,
  LANDSLIDE_TRIGGER     VARCHAR
);

-- Landslide size dimension
CREATE OR REPLACE TABLE SILVER.DIM_LANDSLIDE_SIZE (
  SIZE_ID  INTEGER AUTOINCREMENT PRIMARY KEY,
  SIZE     VARCHAR
);

-- Landslide setting dimension
CREATE OR REPLACE TABLE SILVER.DIM_LANDSLIDE_SETTING (
  SETTING_ID  INTEGER AUTOINCREMENT PRIMARY KEY,
  SETTING     VARCHAR
);

-- Storm dimension 
CREATE OR REPLACE TABLE SILVER.DIM_STORM (
  STORM_ID      INTEGER AUTOINCREMENT PRIMARY KEY,
  STORM_NAME    VARCHAR
);

-- Gazetteer dimension
CREATE OR REPLACE TABLE SILVER.DIM_GAZETTEER (
  GAZETTEER_ID  INTEGER AUTOINCREMENT PRIMARY KEY,
  GAZEETER_CLOSEST_POINT VARCHAR
);

-- Fact table
CREATE OR REPLACE TABLE SILVER.FACT_LANDSLIDE (
  -- This is the fact table's own primary key
  LANDSLIDE_EVENT_KEY       INTEGER AUTOINCREMENT PRIMARY KEY,

  -- FOREIGN KEYS to the dimensions
  
  -- Role-Playing Date Keys
  EVENT_DATE_ID            INTEGER,
  SUBMITTED_DATE_ID        INTEGER,
  CREATED_DATE_ID          INTEGER,
  LAST_EDITED_DATE_ID      INTEGER,

  -- Other Dimension Keys
  SOURCE_ID                INTEGER,
  COUNTRY_ID               INTEGER,
  ADMIN_DIVISION_ID        INTEGER,
  DISTANCE_ID              INTEGER, 
  CATEGORY_ID              INTEGER,
  TRIGGER_ID               INTEGER,
  SIZE_ID                  INTEGER,
  SETTING_ID               INTEGER,
  STORM_ID                 INTEGER,
  GAZETTEER_ID             INTEGER,

  -- These are the numeric values 
  FATALITY_COUNT            NUMBER,
  INJURY_COUNT              NUMBER,
  LONGITUDE                 FLOAT,
  LATITUDE                  FLOAT,
  GAZETEER_DISTANCE         NUMBER,
  ADMIN_DIVISION_POPULATION NUMBER,
  
  -- These are descriptive fields specific to the event
  EVENT_ID                  VARCHAR, -- The original source event_id
  EVENT_TITLE               VARCHAR,
  EVENT_DESCRIPTION         VARCHAR,
  LOCATION_DESCRIPTION      VARCHAR,
  NOTES                     VARCHAR,
  PHOTO_LINK                VARCHAR,
  SOURCE_LINK               VARCHAR,
  SILVER_LOAD_TS            TIMESTAMP_LTZ DEFAULT CURRENT_TIMESTAMP, -- for tracking purposes

  -- Foreign Key Constraints
  CONSTRAINT fk_source FOREIGN KEY (SOURCE_ID) REFERENCES Silver.DIM_SOURCE(SOURCE_ID),
  CONSTRAINT fk_event_date FOREIGN KEY (EVENT_DATE_ID) REFERENCES Silver.DIM_DATE(DATE_ID),
  CONSTRAINT fk_submitted_date FOREIGN KEY (SUBMITTED_DATE_ID) REFERENCES Silver.DIM_DATE(DATE_ID),
  CONSTRAINT fk_created_date FOREIGN KEY (CREATED_DATE_ID) REFERENCES Silver.DIM_DATE(DATE_ID),
  CONSTRAINT fk_edited_date FOREIGN KEY (LAST_EDITED_DATE_ID) REFERENCES Silver.DIM_DATE(DATE_ID),
  CONSTRAINT fk_country FOREIGN KEY (COUNTRY_ID) REFERENCES Silver.DIM_COUNTRY(COUNTRY_ID),
  CONSTRAINT fk_admin_division FOREIGN KEY (ADMIN_DIVISION_ID) REFERENCES Silver.DIM_ADMIN_DIVISION(ADMIN_DIVISION_ID),
  CONSTRAINT fk_distance FOREIGN KEY (DISTANCE_ID) REFERENCES Silver.DIM_DISTANCE(DISTANCE_ID),
  CONSTRAINT fk_category FOREIGN KEY (CATEGORY_ID) REFERENCES Silver.DIM_LANDSLIDE_CATEGORY(CATEGORY_ID),
  CONSTRAINT fk_trigger FOREIGN KEY (TRIGGER_ID) REFERENCES Silver.DIM_LANDSLIDE_TRIGGER(TRIGGER_ID),
  CONSTRAINT fk_size FOREIGN KEY (SIZE_ID) REFERENCES Silver.DIM_LANDSLIDE_SIZE(SIZE_ID),
  CONSTRAINT fk_setting FOREIGN KEY (SETTING_ID) REFERENCES Silver.DIM_LANDSLIDE_SETTING(SETTING_ID),
  CONSTRAINT fk_storm FOREIGN KEY (STORM_ID) REFERENCES Silver.DIM_STORM(STORM_ID),
  CONSTRAINT fk_gazetteer FOREIGN KEY (GAZETTEER_ID) REFERENCES Silver.DIM_GAZETTEER(GAZETTEER_ID)
  
);

### Task 5: Write appropriate SQL to populate silver layer data from bronze layer maintaining data consistency

In [ ]:
-- Step 1: We use -1 as the key for all unknown/N/A/NULL records
INSERT INTO SILVER.DIM_DATE (DATE_ID, DATE, YEAR, MONTH, DAY, DAY_OF_WEEK, QUARTER) 
VALUES (-1, '1900-01-01', 1900, 1, 1, 'N/A', 'N/A');

INSERT INTO SILVER.DIM_SOURCE (SOURCE_ID, SOURCE_NAME) 
VALUES (-1, 'Unknown');

INSERT INTO SILVER.DIM_COUNTRY (COUNTRY_ID, COUNTRY_NAME, COUNTRY_CODE) 
VALUES (-1, 'Unknown', 'N/A');

INSERT INTO SILVER.DIM_ADMIN_DIVISION (ADMIN_DIVISION_ID, ADMIN_DIVISION_NAME) 
VALUES (-1, 'Unknown');

INSERT INTO SILVER.DIM_DISTANCE (DISTANCE_ID, LOCATION_ACCURACY) 
VALUES (-1, 'Unknown');

INSERT INTO SILVER.DIM_LANDSLIDE_CATEGORY (CATEGORY_ID, CATEGORY) 
VALUES (-1, 'Unknown');

INSERT INTO SILVER.DIM_LANDSLIDE_TRIGGER (TRIGGER_ID, LANDSLIDE_TRIGGER) 
VALUES (-1, 'Unknown');

INSERT INTO SILVER.DIM_LANDSLIDE_SIZE (SIZE_ID, SIZE) 
VALUES (-1, 'Unknown');

INSERT INTO SILVER.DIM_LANDSLIDE_SETTING (SETTING_ID, SETTING) 
VALUES (-1, 'Unknown');

INSERT INTO SILVER.DIM_STORM (STORM_ID, STORM_NAME) 
VALUES (-1, 'Unknown');

INSERT INTO SILVER.DIM_GAZETTEER (GAZETTEER_ID, GAZEETER_CLOSEST_POINT) 
VALUES (-1, 'Unknown');

In [ ]:
-- Step 2: Populate the dim tables 
INSERT INTO SILVER.DIM_DATE (DATE, YEAR, MONTH, DAY, DAY_OF_WEEK, QUARTER)
WITH all_raw_dates AS (
  SELECT DISTINCT TRIM(event_date)        AS raw_date_string FROM BRONZE.LANDSLIDE_RAW WHERE event_date IS NOT NULL
  UNION
  SELECT DISTINCT TRIM(submitted_date)    AS raw_date_string FROM BRONZE.LANDSLIDE_RAW WHERE submitted_date IS NOT NULL
  UNION
  SELECT DISTINCT TRIM(created_date)      AS raw_date_string FROM BRONZE.LANDSLIDE_RAW WHERE created_date IS NOT NULL
  UNION
  SELECT DISTINCT TRIM(last_edited_date)  AS raw_date_string FROM BRONZE.LANDSLIDE_RAW WHERE last_edited_date IS NOT NULL
),
parsed_dates AS (
  SELECT
    raw_date_string,
    -- take the date portion before any space (removes time portion: "8/1/08 0:00" -> "8/1/08")
    SPLIT_PART(raw_date_string, ' ', 1) AS date_token,
    -- try several patterns (two-digit year and four-digit year, single/double digit)
    COALESCE(
      TRY_TO_DATE(SPLIT_PART(raw_date_string,' ',1), 'M/D/YY'),
      TRY_TO_DATE(SPLIT_PART(raw_date_string,' ',1), 'MM/DD/YY'),
      TRY_TO_DATE(SPLIT_PART(raw_date_string,' ',1), 'M/D/YYYY'),
      TRY_TO_DATE(SPLIT_PART(raw_date_string,' ',1), 'MM/DD/YYYY'),
      -- as a fallback, try parsing entire token as timestamp and cast to date
      TRY_TO_DATE(TO_VARCHAR(TRY_TO_TIMESTAMP(raw_date_string)), 'YYYY-MM-DD')
    ) AS parsed_date
  FROM all_raw_dates
)
SELECT
  DISTINCT 
  pd.parsed_date AS DATE,
  YEAR(pd.parsed_date) AS YEAR,
  MONTH(pd.parsed_date) AS MONTH,
  DAY(pd.parsed_date) AS DAY,
  DAYNAME(pd.parsed_date) AS DAY_OF_WEEK,
  'Q' || TO_VARCHAR(QUARTER(pd.parsed_date)) AS QUARTER
FROM parsed_dates pd
LEFT JOIN SILVER.DIM_DATE d ON d.DATE = pd.parsed_date
WHERE pd.parsed_date IS NOT NULL
  AND d.DATE IS NULL;

INSERT INTO SILVER.DIM_SOURCE (SOURCE_NAME)
SELECT DISTINCT source_name 
FROM BRONZE.LANDSLide_RAW 
WHERE source_name IS NOT NULL AND source_name NOT IN (SELECT SOURCE_NAME FROM SILVER.DIM_SOURCE);

INSERT INTO SILVER.DIM_COUNTRY (COUNTRY_NAME, COUNTRY_CODE)
SELECT DISTINCT country_name, country_code
FROM BRONZE.LANDSLIDE_RAW 
WHERE country_name IS NOT NULL AND country_name NOT IN (SELECT COUNTRY_NAME FROM SILVER.DIM_COUNTRY);

INSERT INTO SILVER.DIM_ADMIN_DIVISION (ADMIN_DIVISION_NAME)
SELECT DISTINCT admin_division_name
FROM BRONZE.LANDSLIDE_RAW 
WHERE admin_division_name IS NOT NULL AND admin_division_name NOT IN (SELECT ADMIN_DIVISION_NAME FROM SILVER.DIM_ADMIN_DIVISION);

INSERT INTO SILVER.DIM_DISTANCE (LOCATION_ACCURACY)
SELECT DISTINCT location_accuracy
FROM BRONZE.LANDSLIDE_RAW 
WHERE location_accuracy IS NOT NULL AND location_accuracy NOT IN (SELECT LOCATION_ACCURACY FROM SILVER.DIM_DISTANCE);

INSERT INTO SILVER.DIM_LANDSLIDE_CATEGORY (CATEGORY)
SELECT DISTINCT landslide_category
FROM BRONZE.LANDSLIDE_RAW 
WHERE landslide_category IS NOT NULL AND landslide_category NOT IN (SELECT CATEGORY FROM SILVER.DIM_LANDSLIDE_CATEGORY);

INSERT INTO SILVER.DIM_LANDSLIDE_TRIGGER (LANDSLIDE_TRIGGER)
SELECT DISTINCT landslide_trigger
FROM BRONZE.LANDSLIDE_RAW 
WHERE landslide_trigger IS NOT NULL AND landslide_trigger NOT IN (SELECT LANDSLIDE_TRIGGER FROM SILVER.DIM_LANDSLIDE_TRIGGER);

INSERT INTO SILVER.DIM_LANDSLIDE_SIZE (SIZE)
SELECT DISTINCT landslide_size
FROM BRONZE.LANDSLIDE_RAW 
WHERE landslide_size IS NOT NULL AND landslide_size NOT IN (SELECT SIZE FROM SILVER.DIM_LANDSLIDE_SIZE);

INSERT INTO SILVER.DIM_LANDSLIDE_SETTING (SETTING)
SELECT DISTINCT landslide_setting
FROM BRONZE.LANDSLIDE_RAW 
WHERE landslide_setting IS NOT NULL AND landslide_setting NOT IN (SELECT SETTING FROM SILVER.DIM_LANDSLIDE_SETTING);

INSERT INTO SILVER.DIM_STORM (STORM_NAME)
SELECT DISTINCT storm_name
FROM BRONZE.LANDSLIDE_RAW 
WHERE storm_name IS NOT NULL AND storm_name NOT IN (SELECT STORM_NAME FROM SILVER.DIM_STORM);

INSERT INTO SILVER.DIM_GAZETTEER (GAZEETER_CLOSEST_POINT)
SELECT DISTINCT gazeteer_closest_point
FROM BRONZE.LANDSLIDE_RAW 
WHERE gazeteer_closest_point IS NOT NULL AND gazeteer_closest_point NOT IN (SELECT GAZEETER_CLOSEST_POINT FROM SILVER.DIM_GAZETTEER);

In [ ]:
-- Step 3: Populate the fact table 

INSERT INTO SILVER.FACT_LANDSLIDE (
    -- Foreign Keys
    EVENT_DATE_ID,
    SUBMITTED_DATE_ID,
    CREATED_DATE_ID,
    LAST_EDITED_DATE_ID,
    SOURCE_ID,
    COUNTRY_ID,
    ADMIN_DIVISION_ID,
    DISTANCE_ID,
    CATEGORY_ID,
    TRIGGER_ID,
    SIZE_ID,
    SETTING_ID,
    STORM_ID,
    GAZETTEER_ID,
    -- Measures
    FATALITY_COUNT,
    INJURY_COUNT,
    LONGITUDE,
    LATITUDE,
    -- Degenerate Dimensions
    EVENT_ID,
    EVENT_TITLE,
    EVENT_DESCRIPTION,
    LOCATION_DESCRIPTION,
    NOTES,
    PHOTO_LINK,
    SOURCE_LINK,
    GAZETEER_DISTANCE,
    ADMIN_DIVISION_POPULATION
)
SELECT
    -- Look up all foreign keys. Use COALESCE to default to -1 (Unknown) if not found.
    COALESCE(d_event.DATE_ID, -1) AS EVENT_DATE_ID,
    COALESCE(d_submit.DATE_ID, -1) AS SUBMITTED_DATE_ID,
    COALESCE(d_create.DATE_ID, -1) AS CREATED_DATE_ID,
    COALESCE(d_edit.DATE_ID, -1) AS LAST_EDITED_DATE_ID,
    COALESCE(dim_src.SOURCE_ID, -1) AS SOURCE_ID,
    COALESCE(dim_country.COUNTRY_ID, -1) AS COUNTRY_ID,
    COALESCE(dim_admin.ADMIN_DIVISION_ID, -1) AS ADMIN_DIVISION_ID,
    COALESCE(dim_dist.DISTANCE_ID, -1) AS DISTANCE_ID,
    COALESCE(dim_cat.CATEGORY_ID, -1) AS CATEGORY_ID,
    COALESCE(dim_trig.TRIGGER_ID, -1) AS TRIGGER_ID,
    COALESCE(dim_size.SIZE_ID, -1) AS SIZE_ID,
    COALESCE(dim_set.SETTING_ID, -1) AS SETTING_ID,
    COALESCE(dim_storm.STORM_ID, -1) AS STORM_ID,
    COALESCE(dim_gaz.GAZETTEER_ID, -1) AS GAZETTEER_ID,

    -- Measures (convert to numbers, default to 0 if conversion fails)
    COALESCE(TRY_TO_NUMBER(raw.fatality_count), 0),
    COALESCE(TRY_TO_NUMBER(raw.injury_count), 0),
    COALESCE(TRY_TO_DOUBLE(raw.LONGITUDE), 0),
    COALESCE(TRY_TO_DOUBLE(raw.LATITUDE), 0),

    -- Degenerate Dimensions (direct copy)
    raw.event_id,
    raw.event_title,
    raw.event_description,
    raw.location_description,
    raw.notes,
    raw.photo_link,
    raw.source_link,
    raw.gazeteer_distance,
    raw.admin_division_population

FROM BRONZE.LANDSLIDE_RAW raw

-- JOIN all dimensions
LEFT JOIN SILVER.DIM_DATE d_event
  ON d_event.DATE = COALESCE(
      TRY_TO_DATE(SPLIT_PART(TRIM(raw.event_date),' ',1), 'M/D/YY'),
      TRY_TO_DATE(SPLIT_PART(TRIM(raw.event_date),' ',1), 'MM/DD/YY'),
      TRY_TO_DATE(SPLIT_PART(TRIM(raw.event_date),' ',1), 'M/D/YYYY'),
      TRY_TO_DATE(SPLIT_PART(TRIM(raw.event_date),' ',1), 'MM/DD/YYYY'),
      TRY_TO_DATE(TO_VARCHAR(TRY_TO_TIMESTAMP(TRIM(raw.event_date))), 'YYYY-MM-DD')
  )

LEFT JOIN SILVER.DIM_DATE d_submit
  ON d_submit.DATE = COALESCE(
      TRY_TO_DATE(SPLIT_PART(TRIM(raw.submitted_date),' ',1), 'M/D/YY'),
      TRY_TO_DATE(SPLIT_PART(TRIM(raw.submitted_date),' ',1), 'MM/DD/YY'),
      TRY_TO_DATE(SPLIT_PART(TRIM(raw.submitted_date),' ',1), 'M/D/YYYY'),
      TRY_TO_DATE(SPLIT_PART(TRIM(raw.submitted_date),' ',1), 'MM/DD/YYYY'),
      TRY_TO_DATE(TO_VARCHAR(TRY_TO_TIMESTAMP(TRIM(raw.submitted_date))), 'YYYY-MM-DD')
  )


LEFT JOIN SILVER.DIM_DATE d_create
  ON d_create.DATE = COALESCE(
      TRY_TO_DATE(SPLIT_PART(TRIM(raw.created_date),' ',1), 'M/D/YY'),
      TRY_TO_DATE(SPLIT_PART(TRIM(raw.created_date),' ',1), 'MM/DD/YY'),
      TRY_TO_DATE(SPLIT_PART(TRIM(raw.created_date),' ',1), 'M/D/YYYY'),
      TRY_TO_DATE(SPLIT_PART(TRIM(raw.created_date),' ',1), 'MM/DD/YYYY'),
      TRY_TO_DATE(TO_VARCHAR(TRY_TO_TIMESTAMP(TRIM(raw.created_date))), 'YYYY-MM-DD')
  )


LEFT JOIN SILVER.DIM_DATE d_edit
  ON d_edit.DATE = COALESCE(
        TRY_TO_DATE(SPLIT_PART(TRIM(raw.last_edited_date),' ',1), 'M/D/YY'), 
        TRY_TO_DATE(SPLIT_PART(TRIM(raw.last_edited_date),' ',1), 'MM/DD/YY'), 
        TRY_TO_DATE(SPLIT_PART(TRIM(raw.last_edited_date),' ',1), 'M/D/YYYY'), 
        TRY_TO_DATE(SPLIT_PART(TRIM(raw.last_edited_date),' ',1), 'MM/DD/YYYY'),     
        TRY_TO_DATE(TO_VARCHAR(TRY_TO_TIMESTAMP(TRIM(raw.last_edited_date))), 'YYYY-MM-DD') 
      )

LEFT JOIN SILVER.DIM_SOURCE dim_src
  ON TRIM(raw.source_name) = dim_src.SOURCE_NAME

LEFT JOIN SILVER.DIM_COUNTRY dim_country
  ON TRIM(raw.country_name) = dim_country.COUNTRY_NAME

LEFT JOIN SILVER.DIM_ADMIN_DIVISION dim_admin
  ON TRIM(raw.admin_division_name) = dim_admin.ADMIN_DIVISION_NAME

LEFT JOIN SILVER.DIM_DISTANCE dim_dist
  ON TRIM(raw.location_accuracy) = dim_dist.LOCATION_ACCURACY

LEFT JOIN SILVER.DIM_LANDSLIDE_CATEGORY dim_cat
  ON TRIM(raw.landslide_category) = dim_cat.CATEGORY

LEFT JOIN SILVER.DIM_LANDSLIDE_TRIGGER dim_trig
  ON TRIM(raw.landslide_trigger) = dim_trig.LANDSLIDE_TRIGGER

LEFT JOIN SILVER.DIM_LANDSLIDE_SIZE dim_size
  ON TRIM(raw.landslide_size) = dim_size.SIZE

LEFT JOIN SILVER.DIM_LANDSLIDE_SETTING dim_set
  ON TRIM(raw.landslide_setting) = dim_set.SETTING

LEFT JOIN SILVER.DIM_STORM dim_storm
  ON TRIM(raw.storm_name) = dim_storm.STORM_NAME

LEFT JOIN SILVER.DIM_GAZETTEER dim_gaz
  ON TRIM(raw.gazeteer_closest_point) = dim_gaz.GAZEETER_CLOSEST_POINT
;

## Task 6: Define 3 use cases for business reporting. i.e. the kind of roll up data business may ask for your data set.

**1. Which countries have the highest landslide fatalities over time?**

Understanding where landslides cause the greatest harm enables:
* Prioritizing disaster-preparedness resources
* Supporting policy makers in high-risk regions
* Identifying vulnerable populations

**2. What are the most common landslide triggers and how severe are they?**

Identifying which triggers lead to the most severe outcomes helps:
* Anticipate landslide-prone conditions
* Guide public warnings and infrastructure planning
* Understand high-impact hazard types

**3. What seasonal patterns exist in landslide occurrences?**

Seasonality helps stakeholders:
* Forecast risky periods
* Optimize monitoring and emergency response staffing
* Understand environmental drivers (rainy seasons, monsoons, snowmelt, etc.)

## Task 7: Design and populate the Gold layer tables to meet those use cases

How many fatalities per country? Which countries have the highest fatalities?

In [ ]:
CREATE OR REPLACE TABLE GOLD.COUNTRY_FATALITIES AS
SELECT

    CASE
        -- If we already know the country, keep it
        WHEN C.COUNTRY_NAME IS NOT NULL AND C.COUNTRY_NAME != 'Unknown' THEN C.COUNTRY_NAME

        -- NEPAL
        WHEN F.location_description LIKE '%Nepal%' OR F.location_description LIKE '%Kathmandu%' OR F.location_description LIKE '%Pokhara%' OR F.location_description LIKE '%Bhaktapur%' OR F.location_description LIKE '%Lalitpur%' OR F.location_description LIKE '%Birgunj%'
            THEN 'Nepal'

        -- INDIA (Comprehensive State/City List)
        WHEN F.location_description LIKE '%India%' OR F.location_description LIKE '%Nagaland%' OR F.location_description LIKE '%Sikkim%' OR F.location_description LIKE '%Assam%' OR F.location_description LIKE '%Manipur%' OR F.location_description LIKE '%Meghalaya%' OR F.location_description LIKE '%Mizoram%' OR F.location_description LIKE '%West Bengal%' OR F.location_description LIKE '%Uttarakhand%' OR F.location_description LIKE '%Himachal Pradesh%' OR F.location_description LIKE '%Jammu and Kashmir%' OR F.location_description LIKE '%Arunachal Pradesh%' OR F.location_description LIKE '%Karnataka%' OR F.location_description LIKE '%Kerala%' OR F.location_description LIKE '%Tamil Nadu%' OR F.location_description LIKE '%Gujarat%' OR F.location_description LIKE '%Uttar Pradesh%' OR F.location_description LIKE '%Maharashtra%' OR F.location_description LIKE '%Mumbai%' OR F.location_description LIKE '%Delhi%' OR F.location_description LIKE '%Kolkata%'
            THEN 'India'

        -- CHINA (Provinces/Regions)
        WHEN F.location_description LIKE '%China%' OR F.location_description LIKE '%Guizhou%' OR F.location_description LIKE '%Fujian%' OR F.location_description LIKE '%Yunnan%' OR F.location_description LIKE '%Tibet%' OR F.location_description LIKE '%Sichuan%' OR F.location_description LIKE '%Jiangxi%' OR F.location_description LIKE '%Hunan%' OR F.location_description LIKE '%Hong Kong%' OR F.location_description LIKE '%Beijing%' OR F.location_description LIKE '%Shanghai%' OR F.location_description LIKE '%Chongqing%'
            THEN 'China'

        -- PHILIPPINES (Major Islands/Cities)
        WHEN F.location_description LIKE '%Philippines%' OR F.location_description LIKE '%Cebu%' OR F.location_description LIKE '%Davao%' OR F.location_description LIKE '%Mindanao%' OR F.location_description LIKE '%Manila%' OR F.location_description LIKE '%Palawan%'
            THEN 'Philippines'

        -- BANGLADESH (Major Cities/Regions)
        WHEN F.location_description LIKE '%Bangladesh%' OR F.location_description LIKE '%Chittagong%' OR F.location_description LIKE '%Dhaka%'
            THEN 'Bangladesh'

        -- VIETNAM (Major Cities/Provinces)
        WHEN F.location_description LIKE '%Vietnam%' OR F.location_description LIKE '%Hanoi%' OR F.location_description LIKE '%Ho Chi Minh City%' OR F.location_description LIKE '%Da Nang%' OR F.location_description LIKE '%Haiphong%' OR F.location_description LIKE '%Can Tho%' OR F.location_description LIKE '%Lao Cai%' OR F.location_description LIKE '%Ha Giang%' OR F.location_description LIKE '%Sapa%'
            THEN 'Vietnam'

        -- USA (Comprehensive State/City List)
        WHEN F.location_description LIKE '%USA%' OR F.location_description LIKE '%US-101%' OR F.location_description LIKE '%California%' OR F.location_description LIKE '%Kentucky%' OR F.location_description LIKE '%Tennessee%' OR F.location_description LIKE '%Vermont%' OR F.location_description LIKE '%Utah%' OR F.location_description LIKE '%Pennsylvania%' OR F.location_description LIKE '%West Virginia%' OR F.location_description LIKE '%Washington%' OR F.location_description LIKE '%Oregon%' OR F.location_description LIKE '%Colorado%' OR F.location_description LIKE '%Montana%' OR F.location_description LIKE '%Ohio%' OR F.location_description LIKE '%Alaska%' OR F.location_description LIKE '%Hawaii%' OR F.location_description LIKE '%Virginia%' OR F.location_description LIKE '%North Carolina%' OR F.location_description LIKE '%New York%' OR F.location_description LIKE '%, KY' OR F.location_description LIKE '%, CA' OR F.location_description LIKE '%, OR' OR F.location_description LIKE '%, WA'
            THEN 'USA'

        -- MEXICO (States/Cities)
        WHEN F.location_description LIKE '%Mexico%' OR F.location_description LIKE '%Oaxaca%' OR F.location_description LIKE '%Veracruz%' OR F.location_description LIKE '%Puebla%'
            THEN 'Mexico'

        -- INDONESIA (Major Islands/Provinces)
        WHEN F.location_description LIKE '%Indonesia%' OR F.location_description LIKE '%Java%' OR F.location_description LIKE '%Sulawesi%' OR F.location_description LIKE '%Gorontalo%' OR F.location_description LIKE '%Sumatra%' OR F.location_description LIKE '%Kalimantan%' OR F.location_description LIKE '%Bali%' OR F.location_description LIKE '%Bogor%'
            THEN 'Indonesia'

        -- PAKISTAN
        WHEN F.location_description LIKE '%Pakistan%' OR F.location_description LIKE '%Kaghan Valley%' OR F.location_description LIKE '%Gilgit Baltistan%'
            THEN 'Pakistan'

        -- UNITED KINGDOM (Countries/Cities)
        WHEN F.location_description LIKE '%United Kingdom%' OR F.location_description LIKE '%England%' OR F.location_description LIKE '%Scotland%' OR F.location_description LIKE '%Northern Ireland%' OR F.location_description LIKE '%Worcestershire%' OR F.location_description LIKE '%Darwen%' OR F.location_description LIKE '%Jurassic Coast%'
            THEN 'United Kingdom'

        -- CANADA (Provinces)
        WHEN F.location_description LIKE '%Canada%' OR F.location_description LIKE '%British Columbia%' OR F.location_description LIKE '%Alberta%' OR F.location_description LIKE '%Quebec%'
            THEN 'Canada'

        -- NEW ZEALAND
        WHEN F.location_description LIKE '%New Zealand%' OR F.location_description LIKE '%Wellington%' OR F.location_description LIKE '%Auckland%' OR F.location_description LIKE '%Kaikoura%'
            THEN 'New Zealand'

        -- Central Asian Countries (Kyrgyzstan and Tajikistan)
        WHEN F.location_description LIKE '%Kyrgyzstan%' OR F.location_description LIKE '%Jalal-Abad%' OR F.location_description LIKE '%Osh%' THEN 'Kyrgyzstan'
        WHEN F.location_description LIKE '%Tajikistan%' OR F.location_description LIKE '%Panjakent%' OR F.location_description LIKE '%Rasht Valley%' THEN 'Tajikistan'

        -- Other Countries
        WHEN F.location_description LIKE '%Trinidad%' OR F.location_description LIKE '%Tobago%' THEN 'Trinidad and Tobago'
        WHEN F.location_description LIKE '%South Africa%' THEN 'South Africa'
        WHEN F.location_description LIKE '%Iran%' THEN 'Iran'
        WHEN F.location_description LIKE '%Sri Lanka%' THEN 'Sri Lanka'
        WHEN F.location_description LIKE '%France%' THEN 'France'
        WHEN F.location_description LIKE '%Kazakhstan%' THEN 'Kazakhstan'
        WHEN F.location_description LIKE '%Madagascar%' THEN 'Madagascar'
        WHEN F.location_description LIKE '%Norway%' THEN 'Norway'
        WHEN F.location_description LIKE '%Solomon Islands%' THEN 'Solomon Islands'
        WHEN F.location_description LIKE '%New Caledonia%' THEN 'New Caledonia'
        WHEN F.location_description LIKE '%Papua New Guinea%' THEN 'Papua New Guinea'
        WHEN F.location_description LIKE '%Rwanda%' THEN 'Rwanda'
        WHEN F.location_description LIKE '%Bhutan%' THEN 'Bhutan'
        WHEN F.location_description LIKE '%Portugal%' THEN 'Portugal'
        WHEN F.location_description LIKE '%French Polynesia%' THEN 'French Polynesia'
        WHEN F.location_description LIKE '%Fiji%' THEN 'Fiji'
        WHEN F.location_description LIKE '%Colombia%' THEN 'Colombia'
        WHEN F.location_description LIKE '%Nigeria%' THEN 'Nigeria'
        WHEN F.location_description LIKE '%Peru%' THEN 'Peru'
        WHEN F.location_description LIKE '%Italy%' THEN 'Italy'
        WHEN F.location_description LIKE '%Belize%' THEN 'Belize'
        WHEN F.location_description LIKE '%Panama%' THEN 'Panama'
        WHEN F.location_description LIKE '%Jamaica%' THEN 'Jamaica'
        WHEN F.location_description LIKE '%Greece%' THEN 'Greece'
        WHEN F.location_description LIKE '%Japan%' THEN 'Japan'
        WHEN F.location_description LIKE '%Lithuania%' THEN 'Lithuania'
        WHEN F.location_description LIKE '%Switzerland%' THEN 'Switzerland'
        WHEN F.location_description LIKE '%Austria%' THEN 'Austria'
        WHEN F.location_description LIKE '%Guatemala%' THEN 'Guatemala'
        WHEN F.location_description LIKE '%Argentina%' THEN 'Argentina'

        ELSE 'Unknown' -- leave if unknown
    END AS COUNTRY,

    SUM(F.FATALITY_COUNT) AS TOTAL_FATALITIES

FROM SILVER.FACT_LANDSLIDE F
JOIN SILVER.DIM_COUNTRY C ON F.COUNTRY_ID = C.COUNTRY_ID

WHERE C.COUNTRY_NAME != 'Unknown'
    OR F.location_description IS NOT NULL
GROUP BY 1
HAVING TOTAL_FATALITIES >= 1  -- Exclude countries with 0 fatalities
ORDER BY 2 DESC;

In [ ]:
-- get the top 10 countries by fatality count
SELECT
    COUNTRY,
    TOTAL_FATALITIES
FROM
    GOLD.COUNTRY_FATALITIES
WHERE
    COUNTRY != 'Unknown' -- Exclude "Unknown" b/c it is ranked 9th
ORDER BY
    TOTAL_FATALITIES DESC
LIMIT 10;

What are the most common landslide (including mudslide and rock falls) triggers and their severity (measured by fatalities)?

In [ ]:
CREATE OR REPLACE TABLE GOLD.TRIGGER_SIZE_FATALITIES AS
SELECT
    T.LANDSLIDE_TRIGGER, 
    
    CASE
        WHEN S.SIZE IN ('small', 'minor', 'very small') THEN 'Small' 
        WHEN S.SIZE IN ('medium', 'moderate') THEN 'Medium'
        WHEN S.SIZE IN ('large', 'very_large', 'catastrophic') THEN 'Large'
        ELSE 'Unknown Size'
    END AS LANDSLIDE_SIZE,

    COUNT(F.EVENT_ID) AS NUM_LANDSLIDES,
    SUM(F.FATALITY_COUNT) AS TOTAL_FATALITIES

FROM 
    SILVER.FACT_LANDSLIDE F

    JOIN SILVER.DIM_LANDSLIDE_TRIGGER T ON F.TRIGGER_ID = T.TRIGGER_ID

    JOIN SILVER.DIM_LANDSLIDE_SIZE S ON F.SIZE_ID = S.SIZE_ID
    
WHERE 
    T.LANDSLIDE_TRIGGER IS NOT NULL 
    
GROUP BY 
    T.LANDSLIDE_TRIGGER,           
    LANDSLIDE_SIZE
    
ORDER BY 
    T.LANDSLIDE_TRIGGER,           
    LANDSLIDE_SIZE;

In [ ]:
WITH TRIGGER_TOTALS AS (
    SELECT 
        LANDSLIDE_TRIGGER,
        LANDSLIDE_SIZE,
        NUM_LANDSLIDES,
        TOTAL_FATALITIES,

        SUM(NUM_LANDSLIDES) OVER (PARTITION BY LANDSLIDE_TRIGGER) AS TRIGGER_GRAND_TOTAL
    FROM 
        GOLD.TRIGGER_SIZE_FATALITIES
    WHERE 
        LANDSLIDE_TRIGGER IS NOT NULL
        AND LOWER(LANDSLIDE_TRIGGER) != 'unknown'
)

SELECT 
    LANDSLIDE_TRIGGER,
    LANDSLIDE_SIZE,
    NUM_LANDSLIDES,
    TOTAL_FATALITIES
FROM 
    TRIGGER_TOTALS
QUALIFY 
    DENSE_RANK() OVER (ORDER BY TRIGGER_GRAND_TOTAL DESC) <= 5
ORDER BY 
    TRIGGER_GRAND_TOTAL DESC,
    LANDSLIDE_SIZE DESC;

When do the most landslides occur seasonally? Monthly?

In [ ]:
CREATE OR REPLACE TABLE GOLD.QUARTERLY_LANDSLIDE_FATALITIES AS
SELECT 
    D.QUARTER,
    COUNT(F.EVENT_ID) AS NUM_LANDSLIDES,
    SUM(F.FATALITY_COUNT) AS TOTAL_FATALITIES
FROM 
    SILVER.FACT_LANDSLIDE F
    JOIN SILVER.DIM_DATE D ON F.EVENT_DATE_ID = D.DATE_ID
GROUP BY 
    D.QUARTER
ORDER BY 
    D.QUARTER;

In [ ]:
SELECT * FROM GOLD.QUARTERLY_LANDSLIDE_FATALITIES 
ORDER BY QUARTER;

In [ ]:
CREATE OR REPLACE TABLE GOLD.MONTHLY_LANDSLIDES_FATALITIES AS
SELECT 
    D.MONTH,
    MONTHNAME(D.DATE) AS MONTH_NAME,
    COUNT(F.EVENT_ID) AS NUM_LANDSLIDES,
    SUM(F.FATALITY_COUNT) AS TOTAL_FATALITIES
FROM 
    SILVER.FACT_LANDSLIDE F
    JOIN SILVER.DIM_DATE D ON F.EVENT_DATE_ID = D.DATE_ID
GROUP BY 
    D.MONTH, 
    MONTH_NAME
ORDER BY 
    D.MONTH;

In [ ]:
SELECT * FROM GOLD.MONTHLY_LANDSLIDES_FATALITIES 
ORDER BY MONTH;

## Task 8: Create a small data file which is of the same format/columns as the original data file but only a handful of rows, and hand populate it with some new data to simulate new incoming data


Load the new dummy raw data file to the stage  
Home -> Ingestion -> Add data -> Load files into a Stage  
File name: Global_Landslide_Data_New_Dummy.csv

## Task 9: Implement appropriate mechanism to load this incremental data automatically into Bronze, Silver and Gold layers. You may have to execute some code manually for this to happen, that is ok.

In [ ]:
-- Step 1: Create the Stream (Change Data Capture)
-- We create the stream before configuring the pipe so it catches the data the moment it lands.

CREATE OR REPLACE STREAM BRONZE.LANDSLIDE_STREAM 
ON TABLE BRONZE.LANDSLIDE_RAW
APPEND_ONLY = TRUE;

In [ ]:
-- Step 2: Configure Snowpipe (Automated Ingestion)
-- We will define a file format first to keep the pipe definition clean, then create the pipe to listen for your specific dummy file.

-- 1. Create a reusable file format based on your previous COPY command
CREATE OR REPLACE FILE FORMAT BRONZE.MY_CSV_FORMAT
  TYPE = 'CSV'
  FIELD_OPTIONALLY_ENCLOSED_BY = '"'
  SKIP_HEADER = 1;

-- 2. Create the Pipe
-- We will trigger this manually using REFRESH later.
CREATE OR REPLACE PIPE BRONZE.LANDSLIDE_PIPE
AS
COPY INTO BRONZE.LANDSLIDE_RAW
FROM @BRONZE.OUR_LOAD_STAGE
FILE_FORMAT = (FORMAT_NAME = BRONZE.MY_CSV_FORMAT)
PATTERN = '.*Global_Landslide_Data_New_Dummy.csv';

In [ ]:
--Step 3: Create the Task (Silver Transformation)
-- It replicates the silver COPY logic. It joins the Stream (new data) with your Dimensions to find the IDs. If a dimension value doesn't exist (e.g., a new Storm Name), it defaults to -1.

CREATE OR REPLACE TASK SILVER.LOAD_FACT_LANDSLIDE_TASK
  WAREHOUSE = ANIMAL_TASK_WH
  SCHEDULE = '1 MINUTE'
WHEN
  SYSTEM$STREAM_HAS_DATA('BRONZE.LANDSLIDE_STREAM')
AS
INSERT INTO SILVER.FACT_LANDSLIDE (
    EVENT_DATE_ID, SUBMITTED_DATE_ID, CREATED_DATE_ID, LAST_EDITED_DATE_ID,
    SOURCE_ID, COUNTRY_ID, ADMIN_DIVISION_ID, DISTANCE_ID,
    CATEGORY_ID, TRIGGER_ID, SIZE_ID, SETTING_ID, STORM_ID, GAZETTEER_ID,
    FATALITY_COUNT, INJURY_COUNT, LONGITUDE,
    LATITUDE, EVENT_ID, EVENT_TITLE, EVENT_DESCRIPTION, LOCATION_DESCRIPTION, NOTES,
    PHOTO_LINK, SOURCE_LINK, GAZETEER_DISTANCE, ADMIN_DIVISION_POPULATION
)
SELECT
    -- Foreign Key Lookups (Defaulting to -1 if new data doesn't match existing dims)
    COALESCE(d_event.DATE_ID, -1),
    COALESCE(d_submit.DATE_ID, -1),
    COALESCE(d_create.DATE_ID, -1),
    COALESCE(d_edit.DATE_ID, -1),
    COALESCE(dim_src.SOURCE_ID, -1),
    COALESCE(dim_country.COUNTRY_ID, -1),
    COALESCE(dim_admin.ADMIN_DIVISION_ID, -1),
    COALESCE(dim_dist.DISTANCE_ID, -1),
    COALESCE(dim_cat.CATEGORY_ID, -1),
    COALESCE(dim_trig.TRIGGER_ID, -1),
    COALESCE(dim_size.SIZE_ID, -1),
    COALESCE(dim_set.SETTING_ID, -1),
    COALESCE(dim_storm.STORM_ID, -1),
    COALESCE(dim_gaz.GAZETTEER_ID, -1),

    -- Measures
    COALESCE(TRY_TO_NUMBER(st.fatality_count), 0),
    COALESCE(TRY_TO_NUMBER(st.injury_count), 0),
    COALESCE(TRY_TO_DOUBLE(st.LONGITUDE), 0),
    COALESCE(TRY_TO_DOUBLE(st.LATITUDE), 0),

    -- Degenerate Dimensions
    st.event_id, st.event_title, st.event_description, st.location_description, 
    st.notes, st.photo_link, st.source_link, st.gazeteer_distance, st.admin_division_population

FROM BRONZE.LANDSLIDE_STREAM st

-- JOIN LOGIC
-- 1. Date Joins
LEFT JOIN SILVER.DIM_DATE d_event ON d_event.DATE = COALESCE(
      TRY_TO_DATE(SPLIT_PART(TRIM(st.event_date),' ',1), 'M/D/YY'),
      TRY_TO_DATE(SPLIT_PART(TRIM(st.event_date),' ',1), 'MM/DD/YY'),
      TRY_TO_DATE(SPLIT_PART(TRIM(st.event_date),' ',1), 'M/D/YYYY'),
      TRY_TO_DATE(SPLIT_PART(TRIM(st.event_date),' ',1), 'MM/DD/YYYY'),
      TRY_TO_DATE(TO_VARCHAR(TRY_TO_TIMESTAMP(TRIM(st.event_date))), 'YYYY-MM-DD')
)
LEFT JOIN SILVER.DIM_DATE d_submit ON d_submit.DATE = COALESCE(
      TRY_TO_DATE(SPLIT_PART(TRIM(st.submitted_date),' ',1), 'M/D/YY'),
      TRY_TO_DATE(SPLIT_PART(TRIM(st.submitted_date),' ',1), 'MM/DD/YY'),
      TRY_TO_DATE(SPLIT_PART(TRIM(st.submitted_date),' ',1), 'M/D/YYYY'),
      TRY_TO_DATE(SPLIT_PART(TRIM(st.submitted_date),' ',1), 'MM/DD/YYYY'),
      TRY_TO_DATE(TO_VARCHAR(TRY_TO_TIMESTAMP(TRIM(st.submitted_date))), 'YYYY-MM-DD')
)
LEFT JOIN SILVER.DIM_DATE d_create ON d_create.DATE = COALESCE(
      TRY_TO_DATE(SPLIT_PART(TRIM(st.created_date),' ',1), 'M/D/YY'),
      TRY_TO_DATE(SPLIT_PART(TRIM(st.created_date),' ',1), 'MM/DD/YY'),
      TRY_TO_DATE(SPLIT_PART(TRIM(st.created_date),' ',1), 'M/D/YYYY'),
      TRY_TO_DATE(SPLIT_PART(TRIM(st.created_date),' ',1), 'MM/DD/YYYY'),
      TRY_TO_DATE(TO_VARCHAR(TRY_TO_TIMESTAMP(TRIM(st.created_date))), 'YYYY-MM-DD')
)
LEFT JOIN SILVER.DIM_DATE d_edit ON d_edit.DATE = COALESCE(
      TRY_TO_DATE(SPLIT_PART(TRIM(st.last_edited_date),' ',1), 'M/D/YY'),
      TRY_TO_DATE(SPLIT_PART(TRIM(st.last_edited_date),' ',1), 'MM/DD/YY'),
      TRY_TO_DATE(SPLIT_PART(TRIM(st.last_edited_date),' ',1), 'M/D/YYYY'),
      TRY_TO_DATE(SPLIT_PART(TRIM(st.last_edited_date),' ',1), 'MM/DD/YYYY'),
      TRY_TO_DATE(TO_VARCHAR(TRY_TO_TIMESTAMP(TRIM(st.last_edited_date))), 'YYYY-MM-DD')
)

-- 2. Dimension Joins
LEFT JOIN SILVER.DIM_SOURCE dim_src ON TRIM(st.source_name) = dim_src.SOURCE_NAME
LEFT JOIN SILVER.DIM_COUNTRY dim_country ON TRIM(st.country_name) = dim_country.COUNTRY_NAME
LEFT JOIN SILVER.DIM_ADMIN_DIVISION dim_admin ON TRIM(st.admin_division_name) = dim_admin.ADMIN_DIVISION_NAME
LEFT JOIN SILVER.DIM_DISTANCE dim_dist ON TRIM(st.location_accuracy) = dim_dist.LOCATION_ACCURACY
LEFT JOIN SILVER.DIM_LANDSLIDE_CATEGORY dim_cat ON TRIM(st.landslide_category) = dim_cat.CATEGORY
LEFT JOIN SILVER.DIM_LANDSLIDE_TRIGGER dim_trig ON TRIM(st.landslide_trigger) = dim_trig.LANDSLIDE_TRIGGER
LEFT JOIN SILVER.DIM_LANDSLIDE_SIZE dim_size ON TRIM(st.landslide_size) = dim_size.SIZE
LEFT JOIN SILVER.DIM_LANDSLIDE_SETTING dim_set ON TRIM(st.landslide_setting) = dim_set.SETTING
LEFT JOIN SILVER.DIM_STORM dim_storm ON TRIM(st.storm_name) = dim_storm.STORM_NAME
LEFT JOIN SILVER.DIM_GAZETTEER dim_gaz ON TRIM(st.gazeteer_closest_point) = dim_gaz.GAZEETER_CLOSEST_POINT

WHERE st.METADATA$ACTION = 'INSERT' 
AND st.METADATA$ISUPDATE = FALSE;

In [ ]:
-- Create the Stored Procedure to update the silver and gold layer
-- we will wrap the logic in a Stored Procedure and trigger it manually.
CREATE OR REPLACE PROCEDURE SILVER.SP_LOAD_FACT_LANDSLIDE()
RETURNS STRING
LANGUAGE SQL
AS
$$
BEGIN
    ---------------------------------------------------------------
    -- STEP 1: AUTOMATED SILVER REFRESH
    -- This takes data from the stream and inserts it into the Fact table
    ---------------------------------------------------------------
    INSERT INTO SILVER.FACT_LANDSLIDE (
        EVENT_DATE_ID, SUBMITTED_DATE_ID, CREATED_DATE_ID, LAST_EDITED_DATE_ID,
        SOURCE_ID, COUNTRY_ID, ADMIN_DIVISION_ID, DISTANCE_ID,
        CATEGORY_ID, TRIGGER_ID, SIZE_ID, SETTING_ID, STORM_ID, GAZETTEER_ID,
        FATALITY_COUNT, INJURY_COUNT, LONGITUDE,
    LATITUDE,
        EVENT_ID, EVENT_TITLE, EVENT_DESCRIPTION, LOCATION_DESCRIPTION, NOTES,
        PHOTO_LINK, SOURCE_LINK, GAZETEER_DISTANCE, ADMIN_DIVISION_POPULATION
    )
    SELECT
        -- Foreign Key Lookups
        COALESCE(d_event.DATE_ID, -1),
        COALESCE(d_submit.DATE_ID, -1),
        COALESCE(d_create.DATE_ID, -1),
        COALESCE(d_edit.DATE_ID, -1),
        COALESCE(dim_src.SOURCE_ID, -1),
        COALESCE(dim_country.COUNTRY_ID, -1),
        COALESCE(dim_admin.ADMIN_DIVISION_ID, -1),
        COALESCE(dim_dist.DISTANCE_ID, -1),
        COALESCE(dim_cat.CATEGORY_ID, -1),
        COALESCE(dim_trig.TRIGGER_ID, -1),
        COALESCE(dim_size.SIZE_ID, -1),
        COALESCE(dim_set.SETTING_ID, -1),
        COALESCE(dim_storm.STORM_ID, -1),
        COALESCE(dim_gaz.GAZETTEER_ID, -1),

        -- Measures
        COALESCE(TRY_TO_NUMBER(st.fatality_count), 0),
        COALESCE(TRY_TO_NUMBER(st.injury_count), 0),
        COALESCE(TRY_TO_DOUBLE(st.LONGITUDE), 0),
        COALESCE(TRY_TO_DOUBLE(st.LATITUDE), 0),

        -- Degenerate Dimensions
        st.event_id, st.event_title, st.event_description, st.location_description, 
        st.notes, st.photo_link, st.source_link, st.gazeteer_distance, st.admin_division_population

    FROM BRONZE.LANDSLIDE_STREAM st
    
    -- JOIN LOGIC
    LEFT JOIN SILVER.DIM_DATE d_event ON d_event.DATE = COALESCE(TRY_TO_DATE(SPLIT_PART(TRIM(st.event_date),' ',1), 'M/D/YY'), TRY_TO_DATE(SPLIT_PART(TRIM(st.event_date),' ',1), 'MM/DD/YY'), TRY_TO_DATE(SPLIT_PART(TRIM(st.event_date),' ',1), 'M/D/YYYY'), TRY_TO_DATE(SPLIT_PART(TRIM(st.event_date),' ',1), 'MM/DD/YYYY'), TRY_TO_DATE(TO_VARCHAR(TRY_TO_TIMESTAMP(TRIM(st.event_date))), 'YYYY-MM-DD'))
    LEFT JOIN SILVER.DIM_DATE d_submit ON d_submit.DATE = COALESCE(TRY_TO_DATE(SPLIT_PART(TRIM(st.submitted_date),' ',1), 'M/D/YY'), TRY_TO_DATE(SPLIT_PART(TRIM(st.submitted_date),' ',1), 'MM/DD/YY'), TRY_TO_DATE(SPLIT_PART(TRIM(st.submitted_date),' ',1), 'M/D/YYYY'), TRY_TO_DATE(SPLIT_PART(TRIM(st.submitted_date),' ',1), 'MM/DD/YYYY'), TRY_TO_DATE(TO_VARCHAR(TRY_TO_TIMESTAMP(TRIM(st.submitted_date))), 'YYYY-MM-DD'))
    LEFT JOIN SILVER.DIM_DATE d_create ON d_create.DATE = COALESCE(TRY_TO_DATE(SPLIT_PART(TRIM(st.created_date),' ',1), 'M/D/YY'), TRY_TO_DATE(SPLIT_PART(TRIM(st.created_date),' ',1), 'MM/DD/YY'), TRY_TO_DATE(SPLIT_PART(TRIM(st.created_date),' ',1), 'M/D/YYYY'), TRY_TO_DATE(SPLIT_PART(TRIM(st.created_date),' ',1), 'MM/DD/YYYY'), TRY_TO_DATE(TO_VARCHAR(TRY_TO_TIMESTAMP(TRIM(st.created_date))), 'YYYY-MM-DD'))
    LEFT JOIN SILVER.DIM_DATE d_edit ON d_edit.DATE = COALESCE(TRY_TO_DATE(SPLIT_PART(TRIM(st.last_edited_date),' ',1), 'M/D/YY'), TRY_TO_DATE(SPLIT_PART(TRIM(st.last_edited_date),' ',1), 'MM/DD/YY'), TRY_TO_DATE(SPLIT_PART(TRIM(st.last_edited_date),' ',1), 'M/D/YYYY'), TRY_TO_DATE(SPLIT_PART(TRIM(st.last_edited_date),' ',1), 'MM/DD/YYYY'), TRY_TO_DATE(TO_VARCHAR(TRY_TO_TIMESTAMP(TRIM(st.last_edited_date))), 'YYYY-MM-DD'))

    LEFT JOIN SILVER.DIM_SOURCE dim_src ON TRIM(st.source_name) = dim_src.SOURCE_NAME
    LEFT JOIN SILVER.DIM_COUNTRY dim_country ON TRIM(st.country_name) = dim_country.COUNTRY_NAME
    LEFT JOIN SILVER.DIM_ADMIN_DIVISION dim_admin ON TRIM(st.admin_division_name) = dim_admin.ADMIN_DIVISION_NAME
    LEFT JOIN SILVER.DIM_DISTANCE dim_dist ON TRIM(st.location_accuracy) = dim_dist.LOCATION_ACCURACY
    LEFT JOIN SILVER.DIM_LANDSLIDE_CATEGORY dim_cat ON TRIM(st.landslide_category) = dim_cat.CATEGORY
    LEFT JOIN SILVER.DIM_LANDSLIDE_TRIGGER dim_trig ON TRIM(st.landslide_trigger) = dim_trig.LANDSLIDE_TRIGGER
    LEFT JOIN SILVER.DIM_LANDSLIDE_SIZE dim_size ON TRIM(st.landslide_size) = dim_size.SIZE
    LEFT JOIN SILVER.DIM_LANDSLIDE_SETTING dim_set ON TRIM(st.landslide_setting) = dim_set.SETTING
    LEFT JOIN SILVER.DIM_STORM dim_storm ON TRIM(st.storm_name) = dim_storm.STORM_NAME
    LEFT JOIN SILVER.DIM_GAZETTEER dim_gaz ON TRIM(st.gazeteer_closest_point) = dim_gaz.GAZEETER_CLOSEST_POINT

    WHERE st.METADATA$ACTION = 'INSERT' 
    AND st.METADATA$ISUPDATE = FALSE;

    ---------------------------------------------------------------
    -- STEP 2: AUTOMATED GOLD REFRESH
    -- We rebuild the Gold tables to reflect the new totals
    ---------------------------------------------------------------

    -- 1. Refresh Country Fatalities
    CREATE OR REPLACE TABLE GOLD.COUNTRY_FATALITIES AS
    SELECT
        CASE
            WHEN C.COUNTRY_NAME IS NOT NULL AND C.COUNTRY_NAME != 'Unknown' THEN C.COUNTRY_NAME
            -- (Abbreviated logic for readability, assuming standard cleanup is sufficient or you can paste the full CASE statement here if needed)
            ELSE C.COUNTRY_NAME 
        END AS COUNTRY,
        SUM(F.FATALITY_COUNT) AS TOTAL_FATALITIES
    FROM SILVER.FACT_LANDSLIDE F
    JOIN SILVER.DIM_COUNTRY C ON F.COUNTRY_ID = C.COUNTRY_ID
    WHERE C.COUNTRY_NAME != 'Unknown'
    GROUP BY 1
    HAVING TOTAL_FATALITIES >= 1
    ORDER BY 2 DESC;

    -- 2. Refresh Trigger & Size Analysis
    CREATE OR REPLACE TABLE GOLD.TRIGGER_SIZE_FATALITIES AS
    SELECT
        T.LANDSLIDE_TRIGGER, 
        CASE
            WHEN S.SIZE IN ('small', 'minor', 'very small') THEN 'Small' 
            WHEN S.SIZE IN ('medium', 'moderate') THEN 'Medium'
            WHEN S.SIZE IN ('large', 'very_large', 'catastrophic') THEN 'Large'
            ELSE 'Unknown Size'
        END AS LANDSLIDE_SIZE,
        COUNT(F.EVENT_ID) AS NUM_LANDSLIDES,
        SUM(F.FATALITY_COUNT) AS TOTAL_FATALITIES
    FROM SILVER.FACT_LANDSLIDE F
    JOIN SILVER.DIM_LANDSLIDE_TRIGGER T ON F.TRIGGER_ID = T.TRIGGER_ID
    JOIN SILVER.DIM_LANDSLIDE_SIZE S ON F.SIZE_ID = S.SIZE_ID
    WHERE T.LANDSLIDE_TRIGGER IS NOT NULL 
    GROUP BY 1, 2;

    -- 3. Refresh Quarterly Stats
    CREATE OR REPLACE TABLE GOLD.QUARTERLY_LANDSLIDE_FATALITIES AS
    SELECT 
        D.QUARTER,
        COUNT(F.EVENT_ID) AS NUM_LANDSLIDES,
        SUM(F.FATALITY_COUNT) AS TOTAL_FATALITIES
    FROM SILVER.FACT_LANDSLIDE F
    JOIN SILVER.DIM_DATE D ON F.EVENT_DATE_ID = D.DATE_ID
    GROUP BY D.QUARTER
    ORDER BY D.QUARTER;

    -- 4. Refresh Monthly Stats
    CREATE OR REPLACE TABLE GOLD.MONTHLY_LANDSLIDES_FATALITIES AS
    SELECT 
        D.MONTH,
        MONTHNAME(D.DATE) AS MONTH_NAME,
        COUNT(F.EVENT_ID) AS NUM_LANDSLIDES,
        SUM(F.FATALITY_COUNT) AS TOTAL_FATALITIES
    FROM SILVER.FACT_LANDSLIDE F
    JOIN SILVER.DIM_DATE D ON F.EVENT_DATE_ID = D.DATE_ID
    GROUP BY D.MONTH, MONTH_NAME
    ORDER BY D.MONTH;
    
    RETURN 'Incremental Load Silver and Gold Complete';
END;
$$;

In [ ]:
-- 2. Check Current Row Counts (Baseline)
SELECT COUNT(*) AS BRONZE_COUNT FROM BRONZE.LANDSLIDE_RAW;

In [ ]:
SELECT COUNT(*) AS SILVER_FACT_COUNT FROM SILVER.FACT_LANDSLIDE;

In [ ]:
--Step 4: Turn it all on (Execution)
-- A. Trigger the pipe to load the dummy file into Bronze
--Trigger the Pipe (The "Incremental Load") Assuming you have uploaded Global_Landslide_Data_New_Dummy.csv to the stage OUR_LOAD_STAGE, run this:
ALTER PIPE BRONZE.LANDSLIDE_PIPE REFRESH;

In [ ]:
-- B. Wait 30 seconds, then verify data is in the stream
SELECT COUNT(*) FROM BRONZE.LANDSLIDE_STREAM;

In [ ]:
-- C. Run the Procedure to load Silver (This replaces the Task)
CALL SILVER.SP_LOAD_FACT_LANDSLIDE();

## Task 10: Record the comparison of some silver or gold dataset e.g. row counts, to show the incremental data got loaded properly.

In [ ]:
-- D. Validate the final data
-- Show row count comparisons; should add 5 new rows
SELECT count(*) FROM BRONZE.LANDSLIDE_RAW;

In [ ]:
SELECT count(*) FROM SILVER.FACT_LANDSLIDE;

In [ ]:
-- Check Gold (and/or check a specific metric you changed)
SELECT * FROM GOLD.QUARTERLY_LANDSLIDE_FATALITIES 
ORDER BY QUARTER;
-- Note changes from the Task 7, increase in number of landslides and total fatalities

## Task 11: Use at least 2 AI SQL functions on the descriptive column in the bronze layer table and store the output as additional outputs on bronze layer table. There is no need to move these new columns to silver and gold layer

In [ ]:
-- Task 11: AI functions on EVENT_DESCRIPTION
USE ROLE ROLE_Team_CLAWJAWCOIL;
USE WAREHOUSE ANIMAL_TASK_WH;
USE DATABASE DB_TEAM_CLAWJAWCOIL;
USE SCHEMA BRONZE;

ALTER TABLE LANDSLIDE_RAW
ADD COLUMN DESCRIPTION_SENTIMENT VARIANT,
    DESCRIPTION_SUMMARY   VARCHAR;


In [ ]:
UPDATE LANDSLIDE_RAW
SET
  DESCRIPTION_SENTIMENT = AI_SENTIMENT(event_description),
  DESCRIPTION_SUMMARY   = SNOWFLAKE.CORTEX.SUMMARIZE(event_description)
WHERE event_description IS NOT NULL
  AND (DESCRIPTION_SENTIMENT IS NULL OR DESCRIPTION_SUMMARY IS NULL);


In [ ]:
SELECT 
    event_id,
    event_description,
    DESCRIPTION_SENTIMENT,
    DESCRIPTION_SUMMARY
FROM LANDSLIDE_RAW
WHERE event_description IS NOT NULL
LIMIT 20;

## Task 12: Build a Cortex Search service to search on the descriptive column in the bronze layer and try few searches

In [ ]:
-- CONTEXT: Set the appropriate role, warehouse, database, and schema
-- so that the Cortex Search service is created in the correct environment.

USE ROLE ROLE_Team_CLAWJAWCOIL;
USE WAREHOUSE ANIMAL_TASK_WH;
USE DATABASE DB_TEAM_CLAWJAWCOIL;
USE SCHEMA BRONZE;

-- Create (or replace) a Cortex Search service on the EVENT_DESCRIPTION field
-- from the BRONZE.LANDSLIDE_RAW table. This service will allow natural
-- language search over the descriptive text, and expose additional columns
-- as searchable/filterable attributes.

CREATE OR REPLACE CORTEX SEARCH SERVICE LANDSLIDE_DESC_SEARCH_SVC
  ON EVENT_DESCRIPTION
  ATTRIBUTES 
      EVENT_ID,
      EVENT_DATE,
      COUNTRY_NAME,
      LANDSLIDE_CATEGORY,
      LANDSLIDE_TRIGGER,
      SOURCE_NAME
  WAREHOUSE = ANIMAL_TASK_WH
  TARGET_LAG = '1 day'
AS
SELECT
  EVENT_ID,
  EVENT_DESCRIPTION,
  EVENT_DATE,
  COUNTRY_NAME,
  LANDSLIDE_CATEGORY,
  LANDSLIDE_TRIGGER,
  SOURCE_NAME
FROM BRONZE.LANDSLIDE_RAW
WHERE EVENT_DESCRIPTION IS NOT NULL;


In [ ]:
SELECT SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
  'DB_TEAM_CLAWJAWCOIL.BRONZE.LANDSLIDE_DESC_SEARCH_SVC',
  '{
     "query": "heavy rain monsoon",
     "columns": ["EVENT_ID","EVENT_DATE","COUNTRY_NAME","LANDSLIDE_TRIGGER","EVENT_DESCRIPTION"],
     "limit": 5
   }'
);

In [ ]:
SELECT SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
  'DB_TEAM_CLAWJAWCOIL.BRONZE.LANDSLIDE_DESC_SEARCH_SVC',
  '{
     "query": "killed buried houses destroyed homeless",
     "columns": ["EVENT_ID","EVENT_DATE","COUNTRY_NAME","LANDSLIDE_TRIGGER","EVENT_DESCRIPTION"],
     "limit": 5
   }'
);

In [ ]:
SELECT SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
  'DB_TEAM_CLAWJAWCOIL.BRONZE.LANDSLIDE_DESC_SEARCH_SVC',
  '{
     "query": "earthquake rockfall road collapse",
     "columns": ["EVENT_ID","EVENT_DATE","COUNTRY_NAME","LANDSLIDE_TRIGGER","EVENT_DESCRIPTION"],
     "limit": 5
   }'
);

In [ ]:
-- Task 12 (helper feature): Create a rule-based flag indicating
-- whether the event description suggests potential human impact.
-- The flag is TRUE if the description contains any of the
-- specified keywords related to death, injury, or displacement.

UPDATE BRONZE.LANDSLIDE_RAW
SET HUMAN_IMPACT_FLAG =
      LOWER(EVENT_DESCRIPTION) LIKE '%kill%'
   OR LOWER(EVENT_DESCRIPTION) LIKE '%dead%'
   OR LOWER(EVENT_DESCRIPTION) LIKE '%buried%'
   OR LOWER(EVENT_DESCRIPTION) LIKE '%injur%'
   OR LOWER(EVENT_DESCRIPTION) LIKE '%evacuat%'
   OR LOWER(EVENT_DESCRIPTION) LIKE '%homeless%'
WHERE EVENT_DESCRIPTION IS NOT NULL;


In [ ]:
-- Quick sanity check: count how many rows fall into each value
-- of HUMAN_IMPACT_FLAG (TRUE/FALSE or NULL).
-- This helps verify that the update worked as expected and gives
-- a rough sense of how many events may involve human impact.

SELECT HUMAN_IMPACT_FLAG,
       COUNT(*) AS ROW_COUNT  
FROM BRONZE.LANDSLIDE_RAW
GROUP BY HUMAN_IMPACT_FLAG;

## Task 13: Build a Cortex Analyst service to query the structured data in Silver layer. Test it with few natural language business questions

In [ ]:
USE DATABASE DB_TEAM_CLAWJAWCOIL;
USE SCHEMA SILVER;

SHOW TABLES;

-- Made cortex analyst service on AI&ML menu